# Assignment: Fine-Tuning BERT on a Kaggle Dataset


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Load IMDb dataset (CSV file)
import pandas as pd
df = pd.read_csv("movie_data.csv")

In [ ]:

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

from transformers import BertTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

print("Columns:", df.columns)

# -------------------------------
# 2. CREATE LABEL COLUMN# -------------------------------
if 'sentiment' in df.columns:
    if df['sentiment'].dtype == 'object':
        df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
    else:
        df['label'] = df['sentiment']
elif 'label' in df.columns:
    pass
else:
    raise Exception("No label column found!")

df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

# Basic cleaning
df['review'] = df['review'].str.lower()
df['review'] = df['review'].str.replace("<br />", " ", regex=True)

# SPEED TRICK (use small data for now)
df = df.sample(5000, random_state=42)

print("Data size after cleaning:", len(df))


# -------------------------------
# 3. SPLIT DATA
# -------------------------------
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'], df['label'], test_size=0.3, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)


# -------------------------------
# 4. TOKENIZER
# -------------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


# -------------------------------
# 5. DATASET CLASS
# -------------------------------
class IMDbDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label)
        }


# Create datasets
train_dataset = IMDbDataset(train_texts, train_labels)
test_dataset = IMDbDataset(test_texts, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=16)


# -------------------------------
# 6. MODEL
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.to(device)


# -------------------------------
# 7. TRAINING WITH PROGRESS BAR
# -------------------------------
optimizer = AdamW(model.parameters(), lr=2e-5)

model.train()

for epoch in range(2):   # you can increase later
    print(f"\nEpoch {epoch+1}")

    loop = tqdm(train_loader, leave=True)

    for batch in loop:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        loop.set_postfix(loss=loss.item())


# -------------------------------
# 8. EVALUATION
# -------------------------------
def evaluate(model, loader):
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            predictions = torch.argmax(logits, dim=1)

            preds.extend(predictions.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(true_labels, preds)
    prec = precision_score(true_labels, preds)
    rec = recall_score(true_labels, preds)
    f1 = f1_score(true_labels, preds)

    print("\nFinal Results:")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")

    cm = confusion_matrix(true_labels, preds)

    sns.heatmap(cm, annot=True, fmt='d')
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.show()


# Run evaluation
evaluate(model, test_loader)